# ⚠️ Troubleshooting Common Notebook Errors (H8L2)

Some students may experience errors such as:

- ❌ Import errors in the H8L2 notebook
- ❌ NumPy compatibility errors

---

## 🔎 Why This Happens

These issues are usually caused by:

- NumPy being upgraded to **version 2.x**
- `mlxtend` not being installed
- Environment or package mismatch

---

## ✅ Required Fix (Using pip)

Please run the following commands in your **Terminal** or **Anaconda Prompt**:

```bash
pip uninstall numpy -y
pip install numpy==1.26.4
pip install --upgrade mlxtend ipywidgets widgetsnbextension
```

---

## 🔄 After Installation

1. Restart **Jupyter Notebook**
2. Go to **Kernel → Restart Kernel**
3. Run all cells again

---

## ⚠️ Important

🚫 **Do NOT upgrade NumPy to 2.x**

Many ML libraries such as:
- `mlxtend`

are not fully compatible with NumPy 2.x yet and may break your environment.

---

## 🔍 Verify NumPy Version

Run this inside your notebook:

```python
import numpy
print(numpy.__version__)
```

It should display:

```
1.26.4
```


In [13]:
pip install "numpy<2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 28.1 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.2
    Uninstalling numpy-2.4.2:
      Successfully uninstalled numpy-2.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.17.1 which is incompatible.
streamlit 1.37.1 requires pandas<3,>=1.3.0, but you have pandas 3.0.1 which is incompatible.
mlxtend 0.24.0 requires numpy>=2.3.5, but you have numpy 1.26.4 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [48]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from itertools import combinations
from sklearn.base import clone

# Generate synthetic binary classification data
X, y = make_classification(n_samples=200, n_features=10, n_informative=5, n_redundant=5, random_state=42)

In [51]:
X.shape, y.shape

((200, 10), (200,))

In [52]:
# Splitting the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

## Forward Selection

In [54]:
# Base classifier to use for feature selection
classifier = RandomForestClassifier(n_estimators=100, random_state=42)

# Forward Selection
sfs_forward = SFS(classifier, 
           k_features=5, 
           forward=True, 
           floating=False, 
           verbose=2,
           scoring='accuracy',
           cv=4)
sfs_forward = sfs_forward.fit(X_train, y_train)

[Parallel(n_jobs=1)]: Done  10 out of  10 | elapsed:    1.9s finished

[2026-02-24 21:17:11] Features: 1/5 -- score: 0.6210881934566146[Parallel(n_jobs=1)]: Done   9 out of   9 | elapsed:    1.8s finished

[2026-02-24 21:17:13] Features: 2/5 -- score: 0.6943456614509247[Parallel(n_jobs=1)]: Done   8 out of   8 | elapsed:    1.5s finished

[2026-02-24 21:17:14] Features: 3/5 -- score: 0.7738264580369844[Parallel(n_jobs=1)]: Done   7 out of   7 | elapsed:    1.3s finished

[2026-02-24 21:17:16] Features: 4/5 -- score: 0.7473328591749645[Parallel(n_jobs=1)]: Done   6 out of   6 | elapsed:    1.1s finished

[2026-02-24 21:17:17] Features: 5/5 -- score: 0.746977240398293

In [55]:
# Listing selected feature indices and corresponding names
selected_features = sfs_forward.k_feature_idx_
print('Selected features:', selected_features)

Selected features: (1, 3, 4, 5, 7)


#### Overview
Forward Selection is a type of sequential feature selection algorithm that starts with no features and adds them one at a time. In each iteration, it adds the feature that improves the model the most until the desired number of features is reached.

#### Implementation 
[Detailed Documentation](https://rasbt.github.io/mlxtend/api_subpackages/mlxtend.feature_selection/#sequentialfeatureselector)
- **SequentialFeatureSelector** from `mlxtend` is used with a RandomForest classifier.
- It progressively adds features to minimize cross-validated prediction errors.
- The process stops when the specified number of features (`k_features`) is selected.

#### Benefits
- Effective in reducing dimensionality.
- Helps in improving model performance by avoiding overfitting.
- Easier to interpret than models trained on full datasets.

#### Considerations
- Computationally expensive as it evaluates many feature combinations.
- The chosen features are dependent on the model and evaluation metric used.


## Backward Elimination

In [61]:
# Backward Elimination
sfs_backward = SFS(classifier, 
           k_features=5, 
           forward=False, 
           floating=False, 
           verbose=2,
           scoring='accuracy',
           cv=4)
sfs_backward = sfs_backward.fit(X_train, y_train)

[Parallel(n_jobs=1)]: Done  10 out of  10 | elapsed:    1.9s finished

[2026-02-24 21:17:19] Features: 9/5 -- score: 0.8129445234708392[Parallel(n_jobs=1)]: Done   9 out of   9 | elapsed:    1.6s finished

[2026-02-24 21:17:21] Features: 8/5 -- score: 0.813122332859175[Parallel(n_jobs=1)]: Done   8 out of   8 | elapsed:    1.5s finished

[2026-02-24 21:17:22] Features: 7/5 -- score: 0.8133001422475108[Parallel(n_jobs=1)]: Done   7 out of   7 | elapsed:    1.3s finished

[2026-02-24 21:17:23] Features: 6/5 -- score: 0.8200568990042675[Parallel(n_jobs=1)]: Done   6 out of   6 | elapsed:    1.1s finished

[2026-02-24 21:17:24] Features: 5/5 -- score: 0.8198790896159318

In [62]:
# Listing selected feature indices and corresponding names
selected_features_backward = sfs_backward.k_feature_idx_
print('Selected features (Backward):', selected_features_backward)

Selected features (Backward): (0, 1, 6, 8, 9)


#### Overview
Backward Elimination starts with all features and removes the least significant feature at each step which improves the model the least. This process is repeated until the specified number of features is reached.

#### Implementation
- Utilizes `SequentialFeatureSelector` from `mlxtend`.
- Operates by removing features to increase a model's performance according to a given metric.
- Stops when the model retains a predefined number of features.

#### Benefits
- Can lead to simpler, more interpretable models that may generalize better.
- Removes irrelevant or less important features that do not contribute to the model accuracy.

#### Considerations
- Like forward selection, it can be computationally expensive.
- The performance may depend heavily on the initial set of features and the classifier used.


## Exhaustive Search

Patience is bitter, but its fruit is sweet

In [65]:
from itertools import combinations
from sklearn.model_selection import cross_val_score

# Exhaustive Search
def exhaustive_search(X, y, model, max_features):
    best_score = 0
    best_combination = None
    for i in range(1, max_features+1):
        for combination in combinations(range(X.shape[1]), i):
            X_subset = X[:, combination]
            score = np.mean(cross_val_score(model, X_subset, y, cv=5, scoring='accuracy'))
            if score > best_score:
                best_score = score
                best_combination = combination
    return best_combination, best_score

In [68]:
# Assuming we test up to 3 features for exhaustiveness due to computational limits
best_combination, best_score = exhaustive_search(X_train, y_train, classifier, 3)
print('Best combination:', best_combination, 'with score:', best_score)

Best combination: (0, 1, 8) with score: 0.8400000000000001


#### Overview
Exhaustive Search, also known as full search, evaluates every possible combination of features to identify the one that performs best according to a specified criterion.

#### Implementation
- Iterates over all possible combinations of features up to a specified limit (`max_features`).
- Evaluates model performance for each combination using cross-validation.

#### Benefits
- Guarantees finding the best subset of features for the model.
- Provides a benchmark against which to measure other feature selection techniques.

#### Considerations
- Computationally infeasible with large numbers of features or large datasets.
- Often limited to small datasets or reduced feature sets to manage computational costs.
